# LLM-Proto: Interactive Inference

Load a trained checkpoint and run prompts interactively.  
Supports **local checkpoints** and **Google Drive** download.

**Steps:**
1. Configure checkpoint source (local path or Google Drive)
2. Load model + tokenizer
3. Run prompts via an interactive chat widget

---

## 1. Install Dependencies

Installs the minimum set of packages needed for **inference only** (no training libraries like `datasets` or `wandb`):

- **`torch`** — PyTorch for running the model forward pass.
- **`tokenizers`** — HuggingFace's Rust-backed tokenizer for encoding prompts and decoding outputs.
- **`ipywidgets`** — interactive UI widgets (sliders, buttons, text areas) for the chat interface.
- **`google-api-python-client`** — (optional) to download checkpoints from Google Drive.

In [ ]:
# ── Cell 1: Install Dependencies ──
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in [
    "torch>=2.1.0", "tokenizers", "safetensors",
    "numpy", "pyyaml", "ipywidgets",
    "google-api-python-client>=2.100.0", "google-auth>=2.23.0",
]:
    install(pkg)

print("All dependencies installed!")

## 2. Imports & Settings

Configure your inference session here:

| Setting | What it controls |
|---------|-----------------|
| `MODEL_SIZE` | Which model architecture to load (`"tiny"`, `"small"`, `"medium"`, `"base"`, `"large"`, or a YAML file path) |
| `CKPT_DIR` / `CKPT_FILE` | Where to find the checkpoint locally |
| `ENABLE_GDRIVE` / `GDRIVE_FOLDER_ID` | If the checkpoint isn't local, download it from Google Drive |
| `DEFAULT_TEMPERATURE` | Controls randomness: 0.1 = very deterministic, 1.5 = very creative |
| `DEFAULT_TOP_K` | Only sample from the top K most likely tokens |
| `DEFAULT_TOP_P` | Nucleus sampling — only consider tokens whose cumulative probability ≤ P |

> **Important:** `MODEL_SIZE` must match the architecture the checkpoint was trained with. Loading a `"tiny"` checkpoint into a `"small"` model (or vice versa) will fail with a shape mismatch error.

In [ ]:
# ── Cell 2: Imports & Settings ──
import os, sys, torch

# Add project root to path
if os.path.basename(os.getcwd()) != "LLM-Proto":
    if "google.colab" in sys.modules:
        !git clone https://github.com/VlSePr/LLM-Proto
        os.chdir("LLM-Proto")

from src.config import ModelConfig, get_model_config, load_model_config
from src.model import TransformerLM
from src.tokenizer import LLMTokenizer
from src.utils import get_device, get_dtype, load_checkpoint
from src.generate import generate_text

# ─────────────────────────────────────────
# Settings
# ─────────────────────────────────────────
MODEL_SIZE       = "tiny"            # "tiny", "small", "medium", "base", "large"
                                      # or path to a model YAML config file
TOKENIZER_PATH   = "tokenizer_data"

# Checkpoint location
CKPT_DIR         = "checkpoints"     # Folder containing .pt files
CKPT_FILE        = "latest.pt"       # Filename: "latest.pt", "best.pt", "step_5000.pt", etc.

# Google Drive (optional — set to download checkpoint from Drive)
ENABLE_GDRIVE        = True
GDRIVE_FOLDER_ID     = "LLM"           # Colab: folder name; Local: Drive folder ID
GDRIVE_CREDENTIALS   = ""           # Service-account JSON (local only)

# Generation defaults
DEFAULT_MAX_TOKENS   = 256
DEFAULT_TEMPERATURE  = 0.8
DEFAULT_TOP_K        = 50
DEFAULT_TOP_P        = 0.9

# ─────────────────────────────────────────

device = get_device()
dtype = get_dtype()
print(f"Device: {device}")
print(f"Dtype: {dtype}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 3. Load Checkpoint & Model

This cell:

1. **Builds the model architecture** from the selected config (no trained weights yet — just the skeleton).
2. **Loads the checkpoint** (`.pt` file) which contains the trained weights, optimizer state, and metadata.
   - First looks locally in `CKPT_DIR`.
   - If not found and `ENABLE_GDRIVE = True`, automatically downloads from Google Drive.
3. **Loads the tokenizer** — path is read from the checkpoint metadata so it always matches the one used during training.
4. **Sets `model.eval()`** — disables dropout and enables inference mode.

The model summary shows the architecture dimensions and total parameter count so you can verify the right model was loaded.

In [ ]:
# ── Cell 3: Load Checkpoint & Model ──
# Loads from CKPT_DIR / CKPT_FILE. If not found and ENABLE_GDRIVE is set,
# downloads the checkpoint from Google Drive automatically.

# Resolve model config
if os.path.isfile(MODEL_SIZE):
    model_config = load_model_config(MODEL_SIZE)
    print(f"Loaded model config from file: {MODEL_SIZE}")
else:
    model_config = get_model_config(MODEL_SIZE)
    print(f"Using preset model config: {MODEL_SIZE}")

# Build model
model = TransformerLM(model_config).to(device)
print(model.summary())

# Load checkpoint
resume_name = CKPT_FILE.removesuffix(".pt")  # load_checkpoint expects name without extension

gdrive_kw = {}
if ENABLE_GDRIVE and GDRIVE_FOLDER_ID:
    gdrive_kw = dict(gdrive_folder_id=GDRIVE_FOLDER_ID,
                     gdrive_credentials_path=GDRIVE_CREDENTIALS)

ckpt = load_checkpoint(CKPT_DIR, resume_name, model, device=device, **gdrive_kw)
print(f"\nCheckpoint loaded: step {ckpt['step']}, loss {ckpt['loss']:.4f}")

# Load tokenizer (from checkpoint metadata if available, otherwise from TOKENIZER_PATH)
tok_path = ckpt.get("train_config", {}).get("tokenizer_path", TOKENIZER_PATH)
tok = LLMTokenizer(tok_path)
print(f"Tokenizer loaded (vocab_size={tok.vocab_size})")

model.eval()
print("\n✓ Model ready for inference!")

## 4. Quick Test — Batch Prompts

Runs three prompts through the model to verify inference works before launching the interactive UI.

**How generation works under the hood:**
1. The prompt is encoded to token IDs with a BOS (beginning-of-sequence) marker.
2. **Prefill phase**: the full prompt is processed in one forward pass, building the KV-cache.
3. **Decode phase**: one new token is generated per step. Only the new token's query vector attends to the cached K/V, making each step O(T) instead of recomputing the full O(T²) attention.
4. Generation stops when `<|eos|>` is produced or `max_new_tokens` is reached.

The quality of the output depends on how long the model was trained. After 100 steps you'll see mostly noise; after 50K+ steps, coherent sentences emerge.

In [ ]:
# ── Cell 4: Quick Test — Batch Prompts ──
# Run a few prompts to verify the model works before going interactive.

test_prompts = [
    "The meaning of life is",
    "Once upon a time",
    "In the year 2050",
]

print("=" * 60)
for prompt in test_prompts:
    text = generate_text(
        model, tok, prompt,
        max_new_tokens=DEFAULT_MAX_TOKENS,
        temperature=DEFAULT_TEMPERATURE,
        top_k=DEFAULT_TOP_K,
        top_p=DEFAULT_TOP_P,
        device=device,
    )
    print(f"Prompt: {prompt}")
    print(f"Output: {text[:300]}")
    print("-" * 60)

## 5. Interactive Chat Interface

A widget-based UI built with `ipywidgets` that lets you:

- **Type prompts** into a text area and click "Generate" (or use the sliders to adjust parameters live).
- **Adjust sampling** in real time:
  - **Max tokens** — how many new tokens to generate (16–1024).
  - **Temperature** — higher = more creative/random, lower = more focused/deterministic.
  - **Top-K** — restricts sampling to the K highest-probability tokens.
  - **Top-P (nucleus)** — restricts sampling to the smallest set of tokens whose probabilities sum to ≤ P.

> The "Clear" button resets the output area. Each generation is independent (no conversation history maintained).

In [ ]:
# ── Cell 5: Interactive Chat Interface ──
# Widget-based UI for running prompts. Works in Jupyter and Colab.

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ── UI Components ──
prompt_input = widgets.Textarea(
    placeholder="Enter your prompt here...",
    layout=widgets.Layout(width="100%", height="80px"),
)
max_tokens_slider = widgets.IntSlider(
    value=DEFAULT_MAX_TOKENS, min=16, max=1024, step=16,
    description="Max tokens:", style={"description_width": "100px"},
    layout=widgets.Layout(width="50%"),
)
temp_slider = widgets.FloatSlider(
    value=DEFAULT_TEMPERATURE, min=0.1, max=2.0, step=0.05,
    description="Temperature:", style={"description_width": "100px"},
    layout=widgets.Layout(width="50%"),
)
topk_slider = widgets.IntSlider(
    value=DEFAULT_TOP_K, min=1, max=200, step=1,
    description="Top-K:", style={"description_width": "100px"},
    layout=widgets.Layout(width="50%"),
)
topp_slider = widgets.FloatSlider(
    value=DEFAULT_TOP_P, min=0.1, max=1.0, step=0.05,
    description="Top-P:", style={"description_width": "100px"},
    layout=widgets.Layout(width="50%"),
)
generate_btn = widgets.Button(
    description="Generate", button_style="primary",
    layout=widgets.Layout(width="120px", height="36px"),
)
clear_btn = widgets.Button(
    description="Clear", button_style="warning",
    layout=widgets.Layout(width="120px", height="36px"),
)
output_area = widgets.Output(
    layout=widgets.Layout(width="100%", border="1px solid #ccc",
                          min_height="200px", max_height="600px",
                          overflow_y="auto", padding="10px"),
)

# ── Event Handlers ──
def on_generate(btn):
    prompt = prompt_input.value.strip()
    if not prompt:
        return
    with output_area:
        print(f"\n{'='*60}")
        print(f"Prompt: {prompt}")
        print(f"(max_tokens={max_tokens_slider.value}, temp={temp_slider.value:.2f}, "
              f"top_k={topk_slider.value}, top_p={topp_slider.value:.2f})")
        print("-" * 60)
        text = generate_text(
            model, tok, prompt,
            max_new_tokens=max_tokens_slider.value,
            temperature=temp_slider.value,
            top_k=topk_slider.value,
            top_p=topp_slider.value,
            device=device,
        )
        print(text)
        print("=" * 60)

def on_clear(btn):
    output_area.clear_output()

def on_enter(change):
    # Generate on Shift+Enter (textarea submits with newline,
    # so we trigger on the button instead)
    pass

generate_btn.on_click(on_generate)
clear_btn.on_click(on_clear)

# ── Layout ──
controls = widgets.HBox([generate_btn, clear_btn])
sliders = widgets.VBox([max_tokens_slider, temp_slider, topk_slider, topp_slider])

ui = widgets.VBox([
    widgets.HTML("<h3>LLM-Proto Inference</h3>"),
    prompt_input,
    controls,
    sliders,
    widgets.HTML("<hr>"),
    output_area,
])

display(ui)

## 6. Load Prompt from File (Optional)

Three ways to provide a long prompt:

1. **Colab file picker** — upload a `.txt` file directly from your machine.
2. **Google Drive path** — uncomment and set the Drive path to a text file.
3. **Local path** — set `PROMPT_FILE` to any local `.txt` file.

This is useful for testing the model on longer contexts (e.g., a paragraph of text) without pasting into the widget. The file content is used as the prompt, and the model generates a continuation.

In [ ]:
# ── Cell 6: (Optional) Load Prompt from File ──
# Upload a .txt file or point to a Google Drive path,
# then use its content as the prompt.

PROMPT_FILE = ""    # Local path to a .txt file, or leave empty

# ── Option A: Upload via Colab file picker ──
if "google.colab" in sys.modules and not PROMPT_FILE:
    from google.colab import files
    print("Upload a .txt file to use as prompt (or skip):")
    try:
        uploaded = files.upload()
        if uploaded:
            fname = list(uploaded.keys())[0]
            PROMPT_FILE = fname
            print(f"Uploaded: {fname}")
    except Exception:
        print("No file uploaded, skipping.")

# ── Option B: Google Drive path ──
# Uncomment and set the path if your prompt file is on Drive:
# PROMPT_FILE = "/content/drive/MyDrive/prompts/my_prompt.txt"  # Colab

# ── Read & generate ──
if PROMPT_FILE and os.path.isfile(PROMPT_FILE):
    with open(PROMPT_FILE, "r", encoding="utf-8") as f:
        file_prompt = f.read().strip()
    print(f"Prompt from file ({len(file_prompt)} chars):")
    print(f"  {file_prompt[:200]}{'...' if len(file_prompt) > 200 else ''}")
    print("-" * 60)

    text = generate_text(
        model, tok, file_prompt,
        max_new_tokens=DEFAULT_MAX_TOKENS,
        temperature=DEFAULT_TEMPERATURE,
        top_k=DEFAULT_TOP_K,
        top_p=DEFAULT_TOP_P,
        device=device,
    )
    print(f"Output:\n{text}")
else:
    if PROMPT_FILE:
        print(f"File not found: {PROMPT_FILE}")
    else:
        print("No prompt file specified. Use the interactive widget above, "
              "or set PROMPT_FILE to a .txt path.")